# 02 Columbus GWR

Select **Runtime → Change runtime type → NVIDIA GPU**, then run all cells
in a fresh session. Each notebook installs its own immutable public package
checkout and verifies every input. No Python-source upload is required.

Package source: `Favioleiva/gpugwr@bd8457396098a963801d64f0949d0ccd860ad7a8`. These links become usable only
after owner-authorized publication. Local tests are not Colab verification.
A failed GPU request or numerical comparison stops execution visibly.
Save the executed notebook and download the output ZIP as evidence.


In [ ]:
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

SOURCE_COMMIT = "bd8457396098a963801d64f0949d0ccd860ad7a8"
REPOSITORY = "https://github.com/Favioleiva/gpugwr.git"
RUN = Path(tempfile.mkdtemp(prefix="gpugwr-"))
CHECKOUT = RUN / "source"
OUTPUT = RUN / "outputs"
OUTPUT.mkdir()
subprocess.run(["git", "init", str(CHECKOUT)], check=True)
subprocess.run(
    [
        "git",
        "-C",
        str(CHECKOUT),
        "remote",
        "add",
        "origin",
        REPOSITORY,
    ],
    check=True,
)
subprocess.run(
    [
        "git",
        "-C",
        str(CHECKOUT),
        "fetch",
        "--depth",
        "1",
        "origin",
        SOURCE_COMMIT,
    ],
    check=True,
)
subprocess.run(
    [
        "git",
        "-C",
        str(CHECKOUT),
        "checkout",
        "--detach",
        "FETCH_HEAD",
    ],
    check=True,
)
os.environ["CUPY_CACHE_DIR"] = str(RUN / "cupy-cache")
with (OUTPUT / "bootstrap.log").open("w", encoding="utf-8") as log:
    completed = subprocess.run(
        [
            sys.executable,
            str(CHECKOUT / "Colab/bootstrap.py"),
            "--checkout",
            str(CHECKOUT),
            "--commit",
            SOURCE_COMMIT,
            "--output",
            str(OUTPUT / "bootstrap.json"),
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
    )
print((OUTPUT / "bootstrap.log").read_text("utf-8")[-12000:])
completed.check_returncode()
from importlib.metadata import version

stale_modules = []
for package_name in ("numpy", "scipy", "pandas", "pyproj"):
    loaded_module = sys.modules.get(package_name)
    if loaded_module is not None:
        loaded_version = getattr(loaded_module, "__version__", None)
        if loaded_version != version(package_name):
            stale_modules.append(package_name)
if stale_modules:
    raise RuntimeError(
        "Restart the Python runtime and rerun all cells; "
        "already-loaded packages changed: " + str(stale_modules)
    )


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits
from gpuGWR import GWRSpatialCache, gpuGWR
from gpugwr.notebook_measurement import environment, measure
from gpugwr.reproducibility import (
    ReproductionAssets,
    export_geography,
    model_options,
    validate_arrays,
    validate_fit,
    write_json,
    zip_outputs,
)

assets = ReproductionAssets(CHECKOUT, RUN / "data-cache")
runtime = environment()
assert runtime["gpu"]["available"], (
    "A genuine GPU is required; no CPU fallback."
)
runtime["source_commit"] = SOURCE_COMMIT
write_json(OUTPUT / "environment.json", runtime)
print(json.dumps(runtime, indent=2))


49 neighborhoods: CRIME (residential burglaries and vehicle thefts per 1,000 households) ~ INC (household income) + HOVAL (housing value), in original units. INC/HOVAL are thousands of dollars. Original arbitrary Cartesian X/Y units; no EPSG is assigned. Bounds 10–49, no standardization.

        All fits use a Gaussian response, intercept, adaptive bisquare kernel and
        AICc bandwidth selection. Global diagnostics and every retained local
        reference array are compared at `atol=1e-10`, `rtol=1e-8`. Bandwidth and
        neighbor decisions require exact equality. References remain unchanged.


In [ ]:
DATASET = "columbus"
frame, specification = assets.load_case(DATASET)
options = model_options(specification)
print(json.dumps(specification, indent=2))
display(frame.head())
with threadpool_limits(limits=1):
    result, timing = measure(
        lambda: gpuGWR(
            frame, **options, engine="gpu", store_matrices=True
        ).fit(),
        "gpu",
    )
assert result.engine == "gpu"
print(result.summary())
display(result.local_results().head(10))
result.export(OUTPUT / DATASET)
export_geography(
    result, frame, specification, assets, OUTPUT / DATASET
)
validation = validate_fit(
    result,
    assets.acquire(specification["reference"]),
    OUTPUT / "canonical_validation.json",
)
write_json(
    OUTPUT / "performance.json",
    {
        **timing,
        "dataset": DATASET,
        "phase": "first_fit_after_bootstrap",
        "configuration": result.configuration,
    },
)
print("Canonical validation:", validation["passed"])
print("GPU wall seconds:", timing["wall_seconds"])


The export contains aligned geographic identifiers and variable names in
CSV/NPZ, JSON metadata, a statistical summary, the validation report and
runtime measurements. Geographic files use only documented coordinates;
no EPSG code is invented. Sampled device memory is a lower bound on peak
usage and includes other processes. This single fit is not a speedup claim.


In [ ]:
archive = zip_outputs(OUTPUT)
print(f"Evidence archive: {archive}")
try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink

    display(FileLink(str(archive)))
else:
    files.download(str(archive))
